# Experiment 01 — HUGS Locality Probe

This Colab runs the first diagnostic experiment for **Interactive Digital Humans / 4D Human Intelligence**.

**Question:** when we perturb a local body joint, how much of the Gaussian human representation changes outside the intended semantic region?

The notebook does not fabricate results. It installs HUGS, prepares the public NeuMan data and pretrained checkpoints, asks you to provide the licensed SMPL neutral model, then runs this repository's locality probe.


## 0. Colab runtime

Choose **Runtime → Change runtime type → GPU** before running the next cell.


In [ ]:
!nvidia-smi
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 1. Clone the research repository and HUGS

HUGS uses git submodules for the Gaussian rasterizer and KNN extension, so `--recursive` is required.


In [ ]:
%cd /content
!rm -rf interactive-digital-humans ml-hugs
!git clone https://github.com/reusahn/interactive-digital-humans.git
!git clone --recursive https://github.com/apple-aiml-research/ml-hugs.git


## 2. Create the HUGS environment

The official HUGS setup was tested with **Python 3.8, PyTorch 1.13.1, CUDA 11.7**. Colab's base environment changes over time, so this notebook installs Miniconda and runs HUGS inside its own pinned environment instead of modifying the Colab kernel.

This step compiles CUDA extensions. If Colab changes its compiler/CUDA compatibility, save the error log. An environment failure is not an experimental result.


In [ ]:
%cd /content
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!bash miniconda.sh -b -p /content/miniconda
!/content/miniconda/bin/conda create -n hugs python=3.8 -y
!/content/miniconda/bin/conda run -n hugs pip install --upgrade pip
!/content/miniconda/bin/conda install -n hugs -y pytorch==1.13.1 torchvision==0.14.1 torchaudio==0.13.1 pytorch-cuda=11.7 -c pytorch -c nvidia
!/content/miniconda/bin/conda run -n hugs pip install fvcore iopath
!/content/miniconda/bin/conda run -n hugs pip install --no-index --no-cache-dir pytorch3d -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py38_cu117_pyt1131/download.html
%cd /content/ml-hugs
!/content/miniconda/bin/conda run -n hugs pip install submodules/diff-gaussian-rasterization
!/content/miniconda/bin/conda run -n hugs pip install submodules/simple-knn
!/content/miniconda/bin/conda run -n hugs pip install -r requirements.txt
!/content/miniconda/bin/conda run -n hugs pip install git+https://github.com/mattloper/chumpy.git


In [ ]:
!/content/miniconda/bin/conda run -n hugs python - <<'PY'
import torch
print('HUGS env torch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
PY


## 3. Download NeuMan data and HUGS pretrained models

These are the public downloads referenced by the official HUGS repository. AMASS is **not required** for this first locality probe.


In [ ]:
%cd /content/ml-hugs
!bash scripts/prepare_data_models.sh
!find . -maxdepth 3 -type d | head -80


## 4. Provide the licensed SMPL model

HUGS requires the **SMPL neutral body model v1.1.0**. SMPL is license-gated, so this notebook does not download or redistribute it.

Download it yourself from the official SMPL site, rename the neutral model to `SMPL_NEUTRAL.pkl`, then upload it below. If you also have `smpl_uv.obj`, you can upload that too.


In [ ]:
from google.colab import files
from pathlib import Path
import shutil
uploaded = files.upload()
smpl_dir = Path('/content/ml-hugs/data/smpl')
smpl_dir.mkdir(parents=True, exist_ok=True)
for name in uploaded:
    src = Path(name)
    if name.endswith('.pkl'):
        dst = smpl_dir / 'SMPL_NEUTRAL.pkl'
    elif name == 'smpl_uv.obj':
        dst = smpl_dir / 'smpl_uv.obj'
    else:
        continue
    shutil.move(str(src), str(dst))
    print('Saved:', dst)
assert (smpl_dir / 'SMPL_NEUTRAL.pkl').exists(), 'Upload the licensed SMPL neutral .pkl file.'


## 5. Locate a pretrained HUGS experiment directory

The probe needs a directory containing `config_train.yaml` and a human checkpoint. Search the downloaded pretrained models below.


In [ ]:
from pathlib import Path
root = Path('/content/ml-hugs')
configs = list(root.rglob('config_train.yaml'))
print('Found', len(configs), 'candidate configs')
for i, p in enumerate(configs[:50]):
    print(i, p.parent)


Choose one **human** or **human_scene** pretrained output directory from the list above. `lab` is a reasonable first sequence if available.


In [ ]:
HUGS_OUTPUT_DIR = ''  # paste one directory printed above
from pathlib import Path
p = Path(HUGS_OUTPUT_DIR)
assert HUGS_OUTPUT_DIR and (p / 'config_train.yaml').exists(), 'Set HUGS_OUTPUT_DIR to a printed pretrained experiment directory.'
print('Using:', p)


## 6. Sanity-check official HUGS evaluation

Before our original experiment, verify that the official pretrained model loads and renders. This establishes that the baseline environment is valid.


In [ ]:
%cd /content/ml-hugs
!/content/miniconda/bin/conda run -n hugs python scripts/evaluate.py -o "$HUGS_OUTPUT_DIR"


## 7. Run our first locality probe

Default perturbation: **left wrist, z-axis, +10°**. The script stores JSON metrics plus NPZ arrays with before/after Gaussian positions and semantic labels.


In [ ]:
%cd /content/interactive-digital-humans/experiments/01-baseline
!/content/miniconda/bin/conda run -n hugs python hugs_locality_probe.py \
  --hugs-root /content/ml-hugs \
  --output-dir "$HUGS_OUTPUT_DIR" \
  --frame 0 \
  --joint left_wrist \
  --axis z \
  --degrees 10 \
  --save-dir /content/probe_results


In [ ]:
import json, glob
files = sorted(glob.glob('/content/probe_results/frame*.json'))
assert files, 'No probe JSON found. Inspect the previous cell for an error.'
with open(files[-1]) as f:
    result = json.load(f)
result


## 8. Save results to Google Drive

Colab runtimes are temporary. Copy experimental outputs to Drive before ending the session.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil, datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
dst = Path('/content/drive/MyDrive/interactive-digital-humans/experiment-01') / stamp
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree('/content/probe_results', dst)
print('Saved to:', dst)


## Next experiment

After one successful run, sweep multiple joints, axes, angles, and frames. Do **not** interpret a single wrist perturbation as evidence for or against the research hypothesis. The first meaningful result is a distribution of locality/leakage across pose and body regions.
